# 압박 상황(풀카운트/최고압박/박빙경기) 피처 실험 노트북

`train_ensemble.py`에 새로 추가한 `is_full_count`, `is_max_pressure`, `is_close_game`이
LightGBM 성능을 개선하는지 확인합니다. (기본으로 항상 켜져 있어 토글 없음 - `hand_matchup`과
동일한 방식으로, 검증 후 안 좋으면 되돌립니다.)

**사전 검증 결과** (구현 전 실측):
- 풀카운트(3B-2S): 성공률 0.525 -> 0.500 (-2.5%p, 7만행)
- 2아웃+풀카운트(최고압박): 0.524 -> 0.509 (-1.5%p, 2.4만행)
- 박빙 경기(득점차<=1): 0.520 -> 0.528 (+0.9%p, 방향은 반대지만 신호 있음)
- (기각됨: 득점권 주자 유무는 거의 무차이, li 분위는 비단조적이라 별도 피처 안 만듦)

**비교 기준선** (팀x팀맞대결+시즌보정까지 반영된 최신 값):
- LightGBM 전체(147만행) OOF Brier: **0.24376**

**이 파일과 `train_ensemble.py`는 같은 폴더에 있어야 아래 import가 동작합니다.**

In [1]:
import sys, os, time
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

sys.path.append(os.getcwd())
from train_ensemble import (
    TARGET_COL, CAT_COLS, build_features, train_lgb,
)

DATA_DIR = "../open/data"

## 1. 데이터 로드 & 피처 생성 (압박 피처는 기본 포함)

In [2]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), encoding="utf-8-sig")
print(train.shape)

train_feat, feat_cols = build_features(train, None)
cat_features = [c for c in CAT_COLS if c in feat_cols]

new_cols = ["is_full_count", "is_max_pressure", "is_close_game"]
print(f"피처 개수: {len(feat_cols)} (신규: {new_cols})")

r = train_feat[TARGET_COL].mean()
baseline_brier = r * (1 - r)
print(f"기준(무정보) Brier = {baseline_brier:.5f}")

(1475092, 49)
피처 개수: 83 (신규: ['is_full_count', 'is_max_pressure', 'is_close_game'])
기준(무정보) Brier = 0.24944


## 2. 전체 데이터로 바로 확인

In [3]:
X_full = train_feat[feat_cols]
y_full = train_feat[TARGET_COL].values

t0 = time.time()
lgb_models_new, lgb_oof_new = train_lgb(X_full, y_full, X_full, cat_features)
brier_new = brier_score_loss(y_full, lgb_oof_new)
print(f"[LightGBM+압박피처] 소요시간: {time.time()-t0:.1f}초")
print(f"[LightGBM+압박피처] OOF Brier (전체): {brier_new:.5f}")
print(f"참고 - 압박 피처 없는 LightGBM 전체 Brier: 0.24376")
print(f"개선폭: {(0.24376 - brier_new) / 0.24376 * 100:.4f}% (양수면 개선)")

  [LGB fold 0] brier=0.24374
  [LGB fold 1] brier=0.24375
  [LGB fold 2] brier=0.24370
  [LGB fold 3] brier=0.24383
  [LGB fold 4] brier=0.24378
[LightGBM+압박피처] 소요시간: 261.9초
[LightGBM+압박피처] OOF Brier (전체): 0.24376
참고 - 압박 피처 없는 LightGBM 전체 Brier: 0.24376
개선폭: -0.0004% (양수면 개선)


## 3. Feature Importance로 실제 활용도 확인

In [4]:
imp_df = pd.DataFrame({
    f"fold{i}": m.feature_importance(importance_type="gain")
    for i, m in enumerate(lgb_models_new)
}, index=feat_cols)
imp_df["mean_gain"] = imp_df[[c for c in imp_df.columns if c.startswith("fold")]].mean(axis=1)
imp_df["share_pct"] = imp_df["mean_gain"] / imp_df["mean_gain"].sum() * 100
imp_df = imp_df.sort_values("mean_gain", ascending=False)

imp_df_ranked = imp_df.reset_index().rename(columns={"index": "feature"})
imp_df_ranked["rank"] = imp_df_ranked.index + 1
print("압박 피처 순위:")
display(imp_df_ranked[imp_df_ranked["feature"].isin(new_cols)][["rank", "feature", "mean_gain", "share_pct"]])

압박 피처 순위:


,rank,feature,mean_gain,share_pct
59,60,is_close_game,848.507774,0.071139
61,62,is_full_count,742.816197,0.062278
66,67,is_max_pressure,188.801328,0.015829
